# 第61课 · 给百万参数上户口——nn.Module 参数注册、Sequential 与模型保存

**目标**：用 `nn.Module` 重建 L57 的 Linear→ReLU→Linear，掌握参数注册与 `state_dict`。

> **原则**：会用 `nn` 仍要懂底层（L54–L58）；API 是映射，不是黑盒借口。

🔗 **Aurora 连接**：Month 2 关键词识别模型（KWS）、Month 3 ASR 微调、Month 5 LLM 都以 `nn.Module` 为骨架——掌握它的注册机制和序列化（serialization）接口是后续一切工作的前提。

← **上一课**　[L60 · autograd 机制](L60_pytorch_autograd.ipynb)

> 上节课学习了 **autograd 机制**：`grad_fn` 计算图、`backward()`、`retain_grad` 与梯度累积，并与手写 `Value` 引擎数值比对。  
> 本课将探讨 **nn.Module 实战**。

## 本课剧情：参数"住"在哪里，决定了模型能不能被保存和加载

L57 的 `MLP` 类能前向、能反向——但有个隐患：**参数是 `Value` 对象，散落在 Python 变量里**。想保存模型到磁盘？得手工收集每个权重。想把模型搬到 GPU？要一个个转。

`nn.Module` 解决了这个问题，像一个"参数户籍系统"：

```python
class MyLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(4, 4))  # ← 登记为参数
    
    def forward(self, x):
        return x @ self.weight
```

只要用 `nn.Parameter` 包裹，`self.weight` 就自动出现在 `model.parameters()`，被优化器发现，被 `state_dict()` 收录，用 `model.to('cuda')` 一键搬到 GPU。

**三个关键工具**：

| 工具 | 作用 |
|---|---|
| `nn.Parameter(tensor)` | 注册为可训练参数 |
| `model.parameters()` | 迭代器，返回所有参数 |
| `model.state_dict()` | 字典，保存/加载模型状态 |

**Aurora 连接**：`aurora.llm.KVCache` 目前是纯 numpy 的推理组件（从零原则，不需要梯度）；训练侧计划中的 `src/aurora/ml/` 模块将全部以 `nn.Module` 为骨架。而 KVCache 里"不参与梯度、但要跟着模型走设备"的键值缓存，在 torch 世界里正对应本课要讲的 `register_buffer()`。

本节任务：实现 `AudioMLP(nn.Module)` — 两层线性 + ReLU，参数总量 = 40×64+64 + 64×10+10 = 3,274。

In [ ]:
import torch
import torch.nn as nn

## 1. `forward()` 与参数自动注册

`nn.Module` 要求子类实现 `forward()`，调用时写 `model(x)` 而不是 `model.forward(x)`——
前者会触发钩子（hooks），后者绕过它们。

参数注册有两种方式：
- 赋值内置层（`nn.Linear`, `nn.Conv1d` …）：参数自动注册
- 赋值裸张量：**必须** 用 `nn.Parameter(tensor)` 包装，否则不会被追踪

```python
# 正确：参数被注册
self.w = nn.Parameter(torch.randn(10, 5))
# 错误：不被追踪，优化器看不到它
self.w = torch.randn(10, 5)
```

### 深潜：为什么 Linear 的权重是 `(out_features, in_features)` 而非 `(in_features, out_features)`？

很多初学者会困惑：`nn.Linear(40, 64)` 输入是 40 维、输出是 64 维，权重应该是 `(40, 64)` 吧？但 PyTorch 存的是 `(64, 40)`——为什么反过来了？

**数学上的真相**：Linear 层实际执行的公式是
$$y = x \cdot W^T + b$$

其中：
- $x$ 是输入，形状 `(batch, in_features)` = `(batch, 40)`
- $W$ 存储在内存中的形状是 `(out_features, in_features)` = `(64, 40)`  
- $W^T$（转置）的形状是 `(in_features, out_features)` = `(40, 64)`
- $y$ 是输出，形状 `(batch, out_features)` = `(batch, 64)`

**具体演算示例**（以 batch=2, in=40, out=64 为例）：

```
输入 x:          (2, 40)  —— 2个样本，每个40维向量
权重 W(存储):    (64, 40)  —— PyTorch 内存中的形状
权重 W^T(转置):  (40, 64)  —— 实际用于乘法的形状

矩阵乘法：
  x @ W^T = (2, 40) @ (40, 64) = (2, 64)
            ↑      ↑  ↑       ↑   ↑      ↑
          batch  in  in     out  batch out
```

这样设计的好处：
1. **内存高效**：权重矩阵按行存储（每一行对应一个输出神经元的权重），访问模式顺序，缓存友好。
2. **梯度计算高效**：反向传播时梯度也可以用高效的批量运算。
3. **参数初始化一致**：Fan-in/Fan-out 初始化规则要求按行来看。

**对比直觉的补充**：你可能想："输入是 40 维，输出是 64 维，那表示'40→64 的变换'应该是 40×64 的矩阵啊。"——没错，逻辑上是这样。但 PyTorch 为了运算效率，把矩阵**转置后再存**，所以内存里看起来是 64×40。这是一个"逻辑形状"与"存储形状"的差异，很常见。

In [ ]:
# 演示：nn.Parameter 注册
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)           # 自动注册 weight + bias
        self.scale = nn.Parameter(torch.ones(1))  # 手动注册
        self.buf = torch.zeros(1)            # 未注册，仅普通张量

    def forward(self, x):
        return self.fc(x) * self.scale

tiny = TinyNet()
param_names = [n for n, _ in tiny.named_parameters()]
print('注册的参数:', param_names)  # fc.weight, fc.bias, scale
print('buf 未出现:', 'buf' not in param_names)

### 技术细节：nn.Module 如何"追踪"子模块属性

当你写 `self.fc = nn.Linear(4, 2)` 时，发生了什么？

`nn.Module` 重写了 Python 的 `__setattr__()` 方法。每次你给 `self` 赋一个对象时，它会自动检查：
- 如果是 `nn.Module` 的子类（如 `nn.Linear`），自动成为"子模块"，参数会被追踪。
- 如果是 `nn.Parameter`，自动成为"参数"。
- 如果是 `torch.Tensor`，不追踪（除非用 `register_buffer()` 显式注册）。
- 其他 Python 对象（int、str、list 等），也不追踪。

```python
# 这些自动被追踪：
self.fc = nn.Linear(4, 2)         # 子模块 → parameters() 能找到 fc.weight、fc.bias
self.scale = nn.Parameter(...)    # Parameter → parameters() 能找到
self.register_buffer('mean', ...) # buffer → state_dict() 能找到，但不在 parameters()

# 这些 NOT 被追踪：
self.buf = torch.zeros(1)         # 普通 Tensor → 无人管理
self.count = 0                    # 普通 Python int → 无人管理
```

**后果**：只有被追踪的对象才能被：
- `model.parameters()` 找到 → 优化器才能更新它
- `model.state_dict()` 保存 → 模型检查点才包含它
- `model.to(device)` 转移 → 设备转换才会应用到它

In [ ]:
# 验证矩阵乘法：权重形状与计算的对应关系
import torch
import torch.nn as nn

linear_layer = nn.Linear(40, 64)
print(f"Linear(40, 64) 的权重形状: {linear_layer.weight.shape}")  # (64, 40)
print(f"Linear(40, 64) 的偏置形状: {linear_layer.bias.shape}")    # (64,)

# 手工验证矩阵乘法
x = torch.randn(2, 40)  # 2个样本，每个40维
W = linear_layer.weight  # 形状 (64, 40)
b = linear_layer.bias    # 形状 (64,)

# PyTorch 内部实际执行的：y = x @ W.T + b
y_manual = x @ W.T + b  # (2, 40) @ (40, 64) + (64,) = (2, 64)
y_actual = linear_layer(x)  # 用 Linear 层直接计算

print(f"\n输入 x 形状: {x.shape}")
print(f"输出 y 形状: {y_actual.shape}")
print(f"手工计算 vs Linear 层结果是否一致: {torch.allclose(y_manual, y_actual)}")

# 解释：W.T 的形状
print(f"\nW.T (转置后) 的形状: {W.T.shape}")  # (40, 64)
print(f"矩阵乘法 (2, 40) @ (40, 64) 结果是 (2, 64) ✓")

### `register_buffer()` — 正确的非参数状态注册方式

`buf` 的反例展示了**不注册**的后果；正确做法是用 `register_buffer()`。它适合这样一种张量：**不需要梯度，但要跟着模型一起搬设备（`.to(device)`），还要出现在 `state_dict()` 里**。

**实际场景示例**：

假设你的模型需要"记住它见过多少个样本"（用于计算移动平均或学习率调度）。这个计数器：
- 不需要梯度（不参与反向传播）
- 需要在保存/加载模型时被保存（check-point 的一部分）
- 需要跟着模型走设备（CPU ↔ GPU）

这正是 `register_buffer()` 的用处：

```python
class ModelWithCounter(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)
        # 用 register_buffer 登记计数器：state_dict 中有它，但没梯度
        self.register_buffer('sample_count', torch.tensor(0, dtype=torch.long))
    
    def forward(self, x):
        self.sample_count += x.shape[0]  # 累加样本数
        return self.linear(x)

model = ModelWithCounter()
print(model.state_dict())  # sample_count 出现在这里
print(list(model.parameters()))  # sample_count 不出现在这里（无梯度）
```

> 区别一览：
>
> | 方式 | `parameters()` | `state_dict()` | 梯度 | 用处 |
> |---|---|---|---|---|
> | `nn.Parameter` | ✅ | ✅ | ✅ | 可训练权重 |
> | `register_buffer()` | ❌ | ✅ | ❌ | 模型状态（计数器、running mean等） |
> | 裸张量属性 | ❌ | ❌ | ❌ | 临时变量、不保存 |

In [ ]:
# 演示：register_buffer — 模型计数器的实际例子
class TrainedSampleCounter(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 2)
        self.scale = nn.Parameter(torch.ones(1))
        
        # 方式1：register_buffer — 计数器被保存，但无梯度
        self.register_buffer('total_samples', torch.tensor(0, dtype=torch.long))
        
        # 方式2：裸张量（反面教材）— 这个不会被保存
        self.unreg_count = torch.tensor(0, dtype=torch.long)

    def forward(self, x):
        # 累计样本数
        self.total_samples += x.shape[0]
        self.unreg_count += x.shape[0]
        return self.fc(x) * self.scale

model_counter = TrainedSampleCounter()

# 运行几次，计数器会增加
x_test = torch.randn(3, 4)
model_counter(x_test)
model_counter(x_test)
print(f"经过2次前向（各3个样本），total_samples = {model_counter.total_samples.item()}")

# 查看 state_dict
state = model_counter.state_dict()
print(f"\nstate_dict 中有什么：")
for key in state.keys():
    print(f"  {key}")
    
print(f"\ntotal_samples 在 state_dict 中？ {'total_samples' in state}")
print(f"unreg_count 在 state_dict 中？ {'unreg_count' in state}")

# 参数迭代器中呢
param_names = [n for n, _ in model_counter.named_parameters()]
print(f"\ntotal_samples 在 parameters() 中？ {'total_samples' in param_names}")

print("\n✅ register_buffer 验证：")
print("   - total_samples 在 state_dict 中（可被保存/加载）")
print("   - total_samples 不在 parameters() 中（无梯度）")
print("   - unreg_count 不被追踪（丢失）")

## 2. `named_parameters()` 与 `state_dict()`

### 什么是"子模块"？

当你写 `self.fc = nn.Linear(4, 2)` 时，`nn.Linear` 对象本身就是一个 `nn.Module`，所以它自动成为当前模块的**子模块**。子模块有自己的参数（weight、bias），这些参数会被自动"递归"发现。

```python
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 5)     # ← 子模块1（Linear）
        self.fc2 = nn.Linear(5, 2)      # ← 子模块2（Linear）
        self.param = nn.Parameter(...)  # ← 直接参数（不是子模块）

# 递归遍历：
# 找到 fc1.weight, fc1.bias（来自子模块 fc1）
# 找到 fc2.weight, fc2.bias（来自子模块 fc2）
# 找到 param（直接参数）
```

**为什么叫"递归"**？因为子模块还可以包含子模块：

```python
class DeepModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(      # ← 子模块（Sequential）
            nn.Linear(20, 10),              # ← Sequential 内部的子模块
            nn.ReLU(),
            nn.Linear(10, 5),
        )
        self.classifier = nn.Linear(5, 2)  # ← 另一个子模块

# 递归遍历能找到所有层次的参数：encoder 内部的 Linear、classifier 的参数等
```

### `named_parameters()` 和 `state_dict()`

`named_parameters()` 递归遍历所有子模块，返回 `(名称, 张量)` 对；常用于统计参数量或做分层学习率。

```python
total = sum(p.numel() for p in model.parameters())
```

`state_dict()` 返回 `OrderedDict`，键为参数名，值为张量——这是保存/加载检查点（checkpoint，ckpt）的标准接口：

```python
torch.save(model.state_dict(), 'ckpt.pt')
model.load_state_dict(torch.load('ckpt.pt'))
```

两个模型结构相同时，可直接把 `state_dict` 从一个传给另一个，实现权重复制。

In [ ]:
# 演示：state_dict 键名与形状
for name, param in tiny.named_parameters():
    print(f'{name:20s}  shape={list(param.shape)}  numel={param.numel()}')

print('\nstate_dict keys:', list(tiny.state_dict().keys()))

### state_dict() 的一个常见误解：只有数字，没有结构信息

`state_dict()` 只保存权重值，**不保存**"这是个什么模型"的信息。看起来像这样：

```python
{
    'fc.weight': tensor([...]),  # 只有数值
    'fc.bias': tensor([...]),
    'scale': tensor([...]),
}
```

**加载时的正确步骤**：
1. **先创建一个空的模型对象**（定义架构）
2. **再从文件加载权重**

```python
# ❌ 错误做法：你无法通过 state_dict 重现模型架构
loaded_dict = torch.load('model.pt')
# 这只是一个字典，不知道该怎么用

# ✅ 正确做法：先定义模型，再加载权重
model = AudioMLP(40, 64, 10)  # 定义架构
model.load_state_dict(torch.load('model.pt'))  # 加载保存的权重
```

**为什么这样设计**？因为 PyTorch 希望保存文件尽可能小和通用。模型的架构应该在代码里明确定义，检查点只负责保存参数值。

## 3. `nn.Sequential`：省掉 `forward()`

`nn.Sequential` 按顺序串联子模块，自动实现 `forward(x) = layer_n(... layer_1(x))`，无需手写循环。子模块以整数索引命名（`0`, `1`, `2` …），也可用 `OrderedDict` 给定字符串名。

```python
net = nn.Sequential(
    nn.Linear(40, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)
# 等价于：x -> Linear -> ReLU -> Linear -> 输出
```

**什么时候适用、什么时候不适用**：

- **✅ 适用**：每一层的输出直接进入下一层的输入，数据流是单一链条。如上面的例子。
- **❌ 不适用**：
  - **有分支**（一个层的输出流向多个下一层）：
    ```python
    # 不能用 Sequential（分支例子）
    x -> Conv -> [Branch A (Conv->ReLU) and Branch B (Conv->ReLU)] -> Concat -> Output
    ```
  - **有跳连**（某层的输入不仅是上一层的输出，还包括更早的层的输出）：
    ```python
    # 不能用 Sequential（跳连例子，ResNet 风格）
    x -> Linear1 -> ReLU -> Linear2 ⊕ Linear1(x) -> Output
                                     ↑（跳连，把原输入加回来）
    ```

在这些复杂情况下，必须手写 `forward()` 来显式控制数据流。

In [ ]:
# 演示：nn.Sequential 前向
seq = nn.Sequential(
    nn.Linear(40, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)
x_demo = torch.randn(8, 40)
print('Sequential 输出形状:', seq(x_demo).shape)  # (8, 10)
print('子模块列表:', list(seq.children()))

In [ ]:
# 演示：Sequential vs 手写 forward — 两种写法完全等价
# 方式1：用 Sequential（推荐，代码简洁）
class MLPv1(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(40, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )
    
    def forward(self, x):
        return self.layers(x)

# 方式2：手写 forward（显式控制，需要代码更多）
class MLPv2(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(40, 64)
        self.relu = nn.ReLU()
        self.l2 = nn.Linear(64, 10)
    
    def forward(self, x):
        x = self.l1(x)
        x = self.relu(x)
        x = self.l2(x)
        return x

# 验证等价性
mlp1 = MLPv1()
mlp2 = MLPv2()

# 复制权重，使参数相同
mlp2.l1.weight.data = mlp1.layers[0].weight.data.clone()
mlp2.l1.bias.data = mlp1.layers[0].bias.data.clone()
mlp2.l2.weight.data = mlp1.layers[2].weight.data.clone()
mlp2.l2.bias.data = mlp1.layers[2].bias.data.clone()

x_test = torch.randn(8, 40)
out1 = mlp1(x_test)
out2 = mlp2(x_test)

print(f"Sequential 版本输出shape: {out1.shape}")
print(f"手写 forward 版本输出shape: {out2.shape}")
print(f"两种写法的输出是否一致: {torch.allclose(out1, out2)}")

print("\n💡 选择建议：")
print("   - 模型结构简单、无分支/跳连 → 用 Sequential（代码少、易读）")
print("   - 模型有分支、跳连或条件分支 → 手写 forward（清晰、灵活）")

## 4. ✏️ 实现 `class AudioMLP(nn.Module)`

**架构**：`Linear(in→hidden) → ReLU → Linear(hidden→out)`

**实现模板**：
```python
class AudioMLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()  # ← 必须！初始化父类 nn.Module 的参数追踪系统
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_features),
        )
    
    def forward(self, x):
        return self.layers(x)
```

### 为什么一定要 `super().__init__()`？

`super().__init__()` 调用的是父类 `nn.Module` 的 `__init__` 方法，它做了以下关键工作：
- 初始化参数追踪系统（`__setattr__` 钩子）
- 设置 `_parameters`、`_buffers`、`_modules` 等内部字典
- 为 `parameters()`、`state_dict()` 等方法做准备

**如果忘了写这一行会怎样**？
```python
# ❌ 错误：忘了 super().__init__()
class BadMLP(nn.Module):
    def __init__(self):
        # 缺少这一行
        self.fc = nn.Linear(40, 64)  # 参数会被标记为普通属性，不被追踪！
    
    def forward(self, x):
        return self.fc(x)

bad = BadMLP()
print(len(list(bad.parameters())))  # 输出 0，权重消失了！
```

所以 `super().__init__()` 必须是 `__init__` 第一行（或至少在赋值任何子模块/参数之前）。

**为什么用 Sequential**：省去手写 `forward()` 里的链式调用，模块顺序即计算顺序。

**验收标准**：

| 检查项 | 期望值 |
|---|---|
| `m(torch.randn(8,40)).shape` | `(8, 10)` |
| `sum(p.numel() for p in m.parameters())` | `3274` |
| `m.layers[0].weight.shape` | `(64, 40)` |

In [ ]:
class AudioMLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        # ✏️ TODO: 定义 self.layers = nn.Sequential(...)
        raise NotImplementedError("TODO: 在 __init__ 中定义 self.layers = nn.Sequential(nn.Linear(in_features, hidden), nn.ReLU(), nn.Linear(hidden, out_features))")

    def forward(self, x):
        # ✏️ TODO: return self.layers(x)
        raise NotImplementedError("TODO: 在 forward 中返回 self.layers(x)")


In [ ]:
try:
    m = AudioMLP(40, 64, 10)
    out = m(torch.randn(8, 40))
    assert out.shape == (8, 10), f'期望 (8,10)，得到 {out.shape}'
    print('✅ AudioMLP 前向输出形状正确:', out.shape)
except (NotImplementedError, TypeError) as e:
    print(f'⚠️ 练习未完成，请先实现 AudioMLP：{e}')

## 5. 参数实验：验证参数总量

对 `AudioMLP(40, 64, 10)` 手算参数数量，我们需要回忆前面的矩阵乘法公式 $y = x \cdot W^T + b$：

**第一层：Linear(40→64)**
- 输入维度：40，输出维度：64
- 权重 $W$ 的存储形状：`(64, 40)` （输出×输入）
- 权重参数数量：$64 \times 40 = 2560$
- 偏置形状：`(64,)`
- 偏置参数数量：$64$
- 本层合计：$2560 + 64 = 2624$

**第二层：Linear(64→10)**
- 输入维度：64，输出维度：10
- 权重形状：`(10, 64)`
- 权重参数数量：$10 \times 64 = 640$
- 偏置参数数量：$10$
- 本层合计：$640 + 10 = 650$

**总参数数量**：$2624 + 650 = 3274$

**为什么是乘而不是加**？因为每个输出神经元都需要一个完整的权重向量来处理输入。如果有 64 个输出神经元，每个需要 40 个权重，总数就是 $64 \times 40$。

### 初始化策略补充：`torch.randn()` vs `torch.zeros()`

- **`torch.randn(shape)`**：从标准正态分布 $\mathcal{N}(0, 1)$ 中随机采样
  - 用途：初始化**可训练参数**（权重、偏置）
  - 原因：随机初始化打破对称性，使神经元学到不同的特征
  
- **`torch.zeros(shape)`**：全 0 张量
  - 用途：初始化**非参数状态**（计数器、缓存等）
  - 原因：这些变量不参与梯度计算，初始值通常无关紧要

`nn.Linear` 内部已经用 `randn` 初始化权重（严格说是 He/Xavier 初始化），所以我们不需要显式调用 `randn`。

In [ ]:
try:
    m = AudioMLP(40, 64, 10)
    total = sum(p.numel() for p in m.parameters())
    print(f'参数总量: {total}')
    assert total == 3274, f'期望 3274，得到 {total}'
    print('✅ 参数总量与手算一致')

    # 查看每层参数明细
    for name, p in m.named_parameters():
        print(f'  {name:25s}  {list(p.shape)}  = {p.numel()}')

    # 实验：hidden=128 时参数数量
    m128 = AudioMLP(40, 128, 10)
    print(f'\nhidden=128 时参数总量: {sum(p.numel() for p in m128.parameters())}')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️ 练习未完成，请先实现 AudioMLP：{e}')

## 本课收束

`AudioMLP.forward()` 通过 `nn.Sequential` 把线性变换和 ReLU 串联，输出形状 `(batch, out_features)`，与手写 `L57` 的数学等价但参数生命周期由 `nn.Module` 统一管理。
`named_parameters()` 和 `state_dict()` 是 Aurora Month 2 KWS 模型保存检查点、Month 3 ASR 加载预训练权重的直接基础。
下一节（L62）将把特征提取器包装进 `torch.utils.data.Dataset`，构建自定义 `__getitem__`，完成音频数据批量加载。

## ✏️ 白板挑战：nn.Module 参数管理手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：`AudioMLP(40, 64, 10)` 共有多少个参数？逐层列出。  
（L1: 40×64+64=2624，L2: 64×10+10=650，共2624+650=3274）

**问 2**：`nn.Parameter` 和普通 `torch.Tensor` 赋值给 `self.x` 有什么区别？  
（Parameter 自动出现在 parameters()；普通 Tensor 不会，不被优化器更新）

**问 3**：`model.state_dict()` 和 `model.parameters()` 有什么区别？  
（state_dict包含所有buffer和parameter，字典格式；parameters()只返回Parameter，迭代器格式）

**问 4**：`model.to('cuda')` 为什么只对注册的 `Parameter` 和 `buffer` 有效？  
（Module只知道通过register_parameter/register_buffer登记的张量；散落变量不在Module管辖范围）

**问 5**：`nn.Sequential([Linear, ReLU, Linear])` 的 `forward(x)` 等价于什么代码？  
（等价于 x1=Linear1(x); x2=ReLU(x1); out=Linear2(x2); return out — 顺序调用每个子模块）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import torch, torch.nn as nn

# 问1：参数总量
try:
    m_q = AudioMLP(40, 64, 10)
    total_q = sum(p.numel() for p in m_q.parameters())
    assert total_q == 3274, f"参数总量={total_q}"
    l1 = m_q.layers[0].weight.numel() + m_q.layers[0].bias.numel()
    l2 = m_q.layers[2].weight.numel() + m_q.layers[2].bias.numel()
    print(f"Q1 ✅  AudioMLP(40,64,10)参数: L1={l1}(40×64+64), L2={l2}(64×10+10), 共{total_q}")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  参数总量=40×64+64+64×10+10=2624+650=3274（待实现后验证）")

# 问2：Parameter vs 普通 Tensor
class _TestMod(nn.Module):
    def __init__(self):
        super().__init__()
        self.p = nn.Parameter(torch.zeros(3))
        self.t = torch.zeros(3)  # 不注册
    def forward(self, x): return x
tm = _TestMod()
param_names = [n for n, _ in tm.named_parameters()]
assert 'p' in param_names and 't' not in param_names
print(f"Q2 ✅  Parameter注册在parameters()中，普通Tensor不在: {param_names}")

# 问3：state_dict vs parameters
class _TestMod2(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.ones(2))
        self.register_buffer('buf', torch.zeros(2))
    def forward(self, x): return x
tm2 = _TestMod2()
sd_keys = list(tm2.state_dict().keys())
param_keys = [n for n, _ in tm2.named_parameters()]
assert 'buf' in sd_keys and 'buf' not in param_keys
print(f"Q3 ✅  state_dict={sd_keys}(含buffer), parameters={param_keys}(仅Parameter)")

# 问4：to()设备迁移（只测CPU→CPU，不依赖CUDA）
try:
    m_q2 = AudioMLP(4, 8, 2)
    m_q2_cpu = m_q2.to('cpu')
    assert all(p.device.type == 'cpu' for p in m_q2_cpu.parameters())
    print(f"Q4 ✅  model.to('cpu')使所有Parameter迁移到CPU")
except (NotImplementedError, TypeError):
    print(f"Q4 ✅  Module.to(device)递归迁移所有注册的Parameter和Buffer（待实现后验证）")

# 问5：Sequential等价展开
seq_q = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
x_q = torch.randn(3, 4)
out_seq = seq_q(x_q)
out_manual = seq_q[2](seq_q[1](seq_q[0](x_q)))
assert torch.allclose(out_seq, out_manual)
print(f"Q5 ✅  Sequential(x)等价于逐层调用，输出shape={tuple(out_seq.shape)}")
print("\n🎉 nn.Module 参数管理白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l61_review = {
    "nn_parameter_understood":  None,  # 理解nn.Parameter让张量进入parameters()？True/False
    "audio_mlp_impl":           None,  # AudioMLP实现正确，shape/参数总量验证通过？True/False
    "state_dict_understood":    None,  # 理解state_dict包含buffer+parameter，用于保存/加载？True/False
    "sequential_forward":       None,  # 理解Sequential.forward = 顺序调用子模块？True/False
    "whiteboard_passed":        None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l61_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l61_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L61 全部通关！进入 L62：Dataset 与 DataLoader')

In [ ]:
# Aurora 连接：KVCache 的"缓存状态"在 torch 世界对应 register_buffer
from aurora.llm.kvcache import KVCache

print("aurora.llm.kvcache.KVCache:", KVCache)
print("KVCache 基类:", [b.__name__ for b in KVCache.__bases__])  # 纯 numpy 实现，不继承 nn.Module
print("若把 KVCache 迁移到 torch：_k/_v 缓存应以 register_buffer 登记——")
print("  无梯度、随 .to(device) 移动、进 state_dict，正是本课表格里 buffer 的三个特征")

---

→ **下一课**　[L62 · Dataset 与 DataLoader](L62_kws_dataset.ipynb)

> 下节课将学习 **Dataset 与 DataLoader**：自定义 `__getitem__`、用 Aurora 自己的 mel 提取器做 `extract_features`，批量加载关键词音频（无网络时自动回退到合成数据）。